# 01 — Exploratory data analysis

Explores the hourly pre-feature graph (`data/processed/esopus_graph.pkl`, built from the deposit's raw
USGS and AORC files by `scripts/01_build_network.py` and `scripts/02_build_graph.py`) and reproduces the
two analyses that shaped the model inputs:

1. **Collinearity** — features with |r| ≥ 0.8 to an earlier feature are removed (`scripts/03`).
2. **Antecedent lags** — dips in the discharge–turbidity cross-variogram set the lags of the engineered
   antecedent-discharge features (17, 197, 2648, 5753, 7387 h).

Loading the graph needs ~10 GB of RAM.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from esopus import paths
from esopus.constants import LABEL_MAP, OUTLET_NODE
from esopus.data import load_graph
from esopus.features.engineering import correlated_features_to_drop
from esopus.features.variogram import VARIOGRAM_NODES, cross_variogram, find_dips
from esopus.plotting.style import apply_style

apply_style()
G = load_graph(paths.PROCESSED_DIR / "esopus_graph.pkl")
x33, y33 = G.nodes[OUTLET_NODE]["x"], G.nodes[OUTLET_NODE]["y"]
print(f"{G.number_of_nodes()} nodes, {x33.shape[1]} candidate features, {len(x33):,} hourly steps")

## Turbidity, discharge and stream power at the outlet

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(y33, color="#2EAF8A", lw=0.6); axes[0].set_ylabel("Turbidity (FNU)")
axes[1].plot(x33["Q (cfs)"], color="#5F5E5A", lw=0.6); axes[1].set_ylabel("Q (cfs)")
axes[2].plot(x33["SSP"], color="#C0392B", lw=0.6); axes[2].set_ylabel("SSP (W m$^{-2}$)")
axes[0].set_title(LABEL_MAP[OUTLET_NODE]); fig.tight_layout()

In [ ]:
gaged = [n for n in G.nodes if G.nodes[n]["y"].notna().any()]
stack = pd.concat([G.nodes[n]["x"][["Q (cfs)", "SSP"]].assign(y=G.nodes[n]["y"]) for n in gaged]).dropna()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ["Q (cfs)", "SSP"]):
    ax.scatter(np.log(stack[col].clip(lower=1e-3)), stack["y"], s=1, alpha=0.1)
    ax.set_xlabel(f"log {col}"); ax.set_ylabel("Turbidity (FNU)")
fig.suptitle("All gaged nodes"); fig.tight_layout()

## Collinearity filter

In [ ]:
drop = correlated_features_to_drop(G)
print(f"{len(drop)} features dropped (|r| >= 0.8 with an earlier feature):")
print(sorted(drop))

## Discharge–turbidity cross-variogram

Site-averaged cross-variogram over the main gages for lags up to one year. Dips (local minima after the first
rise to the sill) indicate timescales at which antecedent discharge re-correlates with turbidity. The
published lags were picked from this curve (computed on the full record, which includes the test period —
see `docs/KNOWN_ISSUES.md`).

In [ ]:
curves = []
for n in VARIOGRAM_NODES:
    lags, g = cross_variogram(G.nodes[n]["x"]["Q (cfs)"], G.nodes[n]["y"], n_lags=269, max_lag_hours=8760)
    curves.append(g)
mean_gamma = np.mean(curves, axis=0)
dips = find_dips(lags, mean_gamma)
fig, ax = plt.subplots(figsize=(11, 4))
for g in curves:
    ax.plot(lags, g, color="#BBBBBB", lw=0.6)
ax.plot(lags, mean_gamma, color="#C0392B", lw=1.8, label="site average")
for L in (17, 197, 2648, 5753, 7387):
    ax.axvline(L, color="#378ADD", ls=":", lw=1)
ax.set_xlabel("Lag (h)"); ax.set_ylabel("Cross-semivariance"); ax.legend()
dips